In [45]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42

rng = np.random.default_rng(SEED)

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Random seed:", SEED)

NumPy version: 2.1.1
Pandas version: 2.2.3
Random seed: 42


In [46]:
N_USERS = 5_000
N_CREATORS = 500
N_CONTENT = 10_000
N_INTERACTIONS = 250_000

print("Users:", N_USERS)
print("Creators:", N_CREATORS)
print("Content:", N_CONTENT)
print("Interactions:", N_INTERACTIONS)

Users: 5000
Creators: 500
Content: 10000
Interactions: 250000


In [47]:
COUNTRIES = [
    "India",
    "United States",
    "United Kingdom",
    "Canada",
    "Australia",
    "Germany",
    "Singapore"
]

AGE_GROUPS = [
    "13-17",
    "18-24",
    "25-34",
    "35-44",
    "45+"
]

GENRES = [
    "Fantasy",
    "Sci-Fi",
    "Comedy",
    "Drama",
    "Horror",
    "Romance",
    "Action",
    "Animation",
    "Documentary",
    "Mystery"
]

CONTENT_TYPES = [
    "mini",
    "cine",
    "image",
    "video"
]

CREATOR_TYPES = [
    "AI Creator",
    "Influencer",
    "Filmmaker",
    "Artist",
    "Community Creator"
]

print("Genres:", GENRES)
print("Content types:", CONTENT_TYPES)

Genres: ['Fantasy', 'Sci-Fi', 'Comedy', 'Drama', 'Horror', 'Romance', 'Action', 'Animation', 'Documentary', 'Mystery']
Content types: ['mini', 'cine', 'image', 'video']


In [48]:
users = pd.DataFrame({
    "user_id": [f"U{i:05d}" for i in range(1, N_USERS + 1)],

    "country": rng.choice(
        COUNTRIES,
        N_USERS
    ),

    "age_group": rng.choice(
        AGE_GROUPS,
        N_USERS,
        p=[0.05, 0.35, 0.30, 0.20, 0.10]
    ),

    "signup_date": pd.to_datetime(
        rng.integers(
            pd.Timestamp("2024-01-01").value // 10**9,
            pd.Timestamp("2025-06-01").value // 10**9,
            N_USERS
        ),
        unit="s"
    )
})

users["following_count"] = rng.poisson(
    lam=20,
    size=N_USERS
)

users["creator_flag"] = rng.random(N_USERS) < 0.08

users.head()

,user_id,country,age_group,signup_date,following_count,creator_flag
0,U00001,India,13-17,2024-02-08 19:21:10,15,True
1,U00002,Germany,25-34,2024-09-16 07:07:10,17,False
2,U00003,Australia,45+,2024-10-21 15:28:07,20,False
3,U00004,Canada,35-44,2025-01-13 15:33:45,13,False
4,U00005,Canada,18-24,2024-10-22 19:41:57,24,False


In [49]:
def generate_preferences(
    rng,
    genres,
    min_genres=1,
    max_genres=4
):
    n = rng.integers(
        min_genres,
        max_genres + 1
    )

    return rng.choice(
        genres,
        size=n,
        replace=False
    ).tolist()


users["preferred_genres"] = [
    generate_preferences(rng, GENRES)
    for _ in range(N_USERS)
]

users[[
    "user_id",
    "preferred_genres"
]].head()

,user_id,preferred_genres
0,U00001,"[Action, Romance, Documentary]"
1,U00002,"[Animation, Documentary, Comedy, Horror]"
2,U00003,"[Documentary, Animation, Comedy, Horror]"
3,U00004,"[Animation, Drama]"
4,U00005,"[Romance, Mystery]"


In [50]:
creators = pd.DataFrame({
    "creator_id": [
        f"C{i:04d}"
        for i in range(1, N_CREATORS + 1)
    ],

    "creator_name": [
        f"Creator_{i:04d}"
        for i in range(1, N_CREATORS + 1)
    ],

    "signup_date": pd.to_datetime(
        rng.integers(
            pd.Timestamp("2024-01-01").value // 10**9,
            pd.Timestamp("2026-06-01").value // 10**9,
            N_CREATORS
        ),
        unit="s"
    ),

    "creator_type": rng.choice(
        CREATOR_TYPES,
        N_CREATORS
    )
})

creators["followers"] = (
    rng.lognormal(
        mean=4.0,
        sigma=1.2,
        size=N_CREATORS
    )
).astype(int)

creators["followers"] = creators["followers"].clip(
    lower=10
)

creators.head()

,creator_id,creator_name,signup_date,creator_type,followers
0,C0001,Creator_0001,2024-07-28 20:15:02,Artist,17
1,C0002,Creator_0002,2024-08-07 17:26:27,Artist,167
2,C0003,Creator_0003,2024-04-12 16:14:44,Influencer,76
3,C0004,Creator_0004,2024-06-26 18:52:05,Influencer,11
4,C0005,Creator_0005,2024-03-09 20:31:20,Community Creator,37


In [51]:
content_created_at = pd.to_datetime(
    rng.integers(
        pd.Timestamp("2025-06-01").value // 10**9,
        pd.Timestamp("2026-08-31").value // 10**9,
        N_CONTENT
    ),
    unit="s"
)

# Assign a creator who had already signed up
content_creator_ids = []

creator_signup_dates = creators.set_index("creator_id")["signup_date"]

for created_at in content_created_at:
    eligible_creators = creator_signup_dates[
        creator_signup_dates <= created_at
    ].index

    content_creator_ids.append(
        rng.choice(eligible_creators)
    )

content = pd.DataFrame({
    "content_id": [
        f"CT{i:06d}"
        for i in range(1, N_CONTENT + 1)
    ],

    "creator_id": content_creator_ids,

    "content_type": rng.choice(
        CONTENT_TYPES,
        N_CONTENT,
        p=[0.45, 0.20, 0.20, 0.15]
    ),

    "genre": rng.choice(
        GENRES,
        N_CONTENT
    ),

    "created_at": content_created_at
})

content.head()

,content_id,creator_id,content_type,genre,created_at
0,CT000001,C0131,mini,Mystery,2025-12-17 12:34:36
1,CT000002,C0484,mini,Horror,2026-08-01 20:42:55
2,CT000003,C0195,mini,Sci-Fi,2026-06-27 15:47:40
3,CT000004,C0328,mini,Romance,2026-04-26 06:39:06
4,CT000005,C0406,image,Action,2025-11-02 17:31:50


In [52]:
content["duration"] = np.where(
    content["content_type"] == "mini",

    rng.lognormal(
        mean=2.5,
        sigma=0.5,
        size=N_CONTENT
    ),

    rng.lognormal(
        mean=4.0,
        sigma=0.7,
        size=N_CONTENT
    )
)

content["duration"] = content["duration"].round(1)

In [53]:
TAGS = [
    "cinematic",
    "ai",
    "creative",
    "trending",
    "story",
    "visual",
    "experimental",
    "viral",
    "character",
    "animation"
]

content["tags"] = [
    ",".join(
        rng.choice(
            TAGS,
            size=rng.integers(1, 4),
            replace=False
        )
    )
    for _ in range(N_CONTENT)
]

content["is_template"] = (
    rng.random(N_CONTENT) < 0.18
)

content["is_recreation"] = (
    rng.random(N_CONTENT) < 0.12
)

content.head()

,content_id,creator_id,content_type,genre,created_at,duration,tags,is_template,is_recreation
0,CT000001,C0131,mini,Mystery,2025-12-17 12:34:36,24.6,"visual,cinematic",False,False
1,CT000002,C0484,mini,Horror,2026-08-01 20:42:55,21.6,"story,ai,animation",False,False
2,CT000003,C0195,mini,Sci-Fi,2026-06-27 15:47:40,14.7,ai,False,False
3,CT000004,C0328,mini,Romance,2026-04-26 06:39:06,6.6,"ai,cinematic,viral",False,False
4,CT000005,C0406,image,Action,2025-11-02 17:31:50,88.7,"trending,visual,story",True,False


In [54]:
print("Users:", users.shape)
print("Creators:", creators.shape)
print("Content:", content.shape)

print("\n--- Creator follower statistics ---")
print(creators["followers"].describe())

print("\n--- Content duration statistics ---")
print(content["duration"].describe())

Users: (5000, 7)
Creators: (500, 5)
Content: (10000, 9)

--- Creator follower statistics ---
count     500.000000
mean      104.708000
std       181.171446
min        10.000000
25%        20.000000
50%        46.000000
75%       109.000000
max      1917.000000
Name: followers, dtype: float64

--- Content duration statistics ---
count    10000.000000
mean        44.429240
std         51.478715
min          2.100000
25%         12.600000
50%         25.700000
75%         58.400000
max        743.500000
Name: duration, dtype: float64


In [55]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           5000 non-null   object        
 1   country           5000 non-null   object        
 2   age_group         5000 non-null   object        
 3   signup_date       5000 non-null   datetime64[ns]
 4   following_count   5000 non-null   int64         
 5   creator_flag      5000 non-null   bool          
 6   preferred_genres  5000 non-null   object        
dtypes: bool(1), datetime64[ns](1), int64(1), object(4)
memory usage: 239.4+ KB


In [56]:
creators.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   creator_id    500 non-null    object        
 1   creator_name  500 non-null    object        
 2   signup_date   500 non-null    datetime64[ns]
 3   creator_type  500 non-null    object        
 4   followers     500 non-null    int64         
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 19.7+ KB


In [57]:
content.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   content_id     10000 non-null  object        
 1   creator_id     10000 non-null  object        
 2   content_type   10000 non-null  object        
 3   genre          10000 non-null  object        
 4   created_at     10000 non-null  datetime64[ns]
 5   duration       10000 non-null  float64       
 6   tags           10000 non-null  object        
 7   is_template    10000 non-null  bool          
 8   is_recreation  10000 non-null  bool          
dtypes: bool(2), datetime64[ns](1), float64(1), object(5)
memory usage: 566.5+ KB


In [58]:
RAW_DATA_PATH = Path("../data/raw")

RAW_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

users.to_csv(
    RAW_DATA_PATH / "users.csv",
    index=False
)

creators.to_csv(
    RAW_DATA_PATH / "creators.csv",
    index=False
)

content.to_csv(
    RAW_DATA_PATH / "content.csv",
    index=False
)

print("Initial datasets saved successfully.")

Initial datasets saved successfully.
